# 04 Hugging Face 기반 AI 에이전트 사례

## 04-2 멀티 태스크 기반 AI 에이전트 사례

### 04-2-4 실습: 멀티 태스크 AI 에이전트 실행

> 실습 목적 “여러 모델을 동시에 쓰는 것”이 아니라 “모델 호출 순서가 에이전트 구조임을 체감  

#### (1) 실습 1. 감성 분류 → 요약 순차 실행  

##### 1) Hugging Face transformers 라이브러리 설치

In [1]:
# -q 옵션: 설치 로그를 간단히 표시
!pip install transformers -q


##### 2) Hugging Face 토큰 발급 및 Colab 보안 저장소에 추가

i.  **Hugging Face 웹사이트에서 토큰 발급**: Hugging Face (huggingface.co) 에 로그인하여 `Settings` -> `Access Tokens` 페이지에서 새 토큰을 생성합니다. (권한은 `read` 이상으로 설정)  

ii.  **Colab 보안 저장소에 추가**: Colab 환경에서 왼쪽 패널의 '🔑' 아이콘(비밀번호 모양)을 클릭하여 `Secret` 탭을 엽니다. `New secret` 버튼을 클릭하여 `Name`에 `HF_TOKEN`을 입력하고, `Value`에 Hugging Face에서 발급받은 토큰 값을 붙여넣습니다. 그리고 'Notebook access'를 켜주세요.

##### 3) 코드에서 Hugging Face 토큰 사용하기

이제 Colab 보안 저장소에 저장된 `HF_TOKEN`을 코드에서 불러와 Hugging Face에 로그인할 수 있습니다. 아래 코드를 실행해 주세요.

In [3]:
from huggingface_hub import login
from google.colab import userdata

# Colab 보안 저장소에서 HF_TOKEN 불러오기
hf_token = userdata.get('HF_TOKEN')


# Hugging Face 로그인
login(token=hf_token)

print("Hugging Face에 성공적으로 로그인했습니다!")

Hugging Face에 성공적으로 로그인했습니다!


##### 4) 멀티 태스크 모델 조합

1.   감성 분류 모델  
2.   요약 모델



In [7]:
from transformers import pipeline

# 1단계: 감성 분류 모델
classifier = pipeline(
    task="text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# 2단계: 요약 모델
summarizer = pipeline(
    task="summarization",
    model="facebook/bart-large-cnn"
)

review = """
The product quality is good, and it meets the expected standards.
The materials and overall build feel reliable and well-designed.
However, the delivery process was much slower than promised.
The package arrived several days later than the estimated date.
There was no clear explanation or advance notice regarding the delay.
This caused inconvenience and uncertainty during the waiting period.
In addition, customer support was unhelpful when contacted.
Responses were slow and did not provide useful information.
The lack of clear communication was disappointing.
Improving delivery speed and customer support would greatly enhance the overall experience.
"""

# 원문에 대한 감성 분류
sentiment = classifier(review)
print("\n\n--- 원문 대상 감성분석 결과 ---\n")
print("Sentiment:", sentiment)

# 요약
summary = summarizer(review, max_length=40, min_length=15, do_sample=False)
print("\n\n--- 텍스트 요약 결과 ---\n")
print("Summary:", summary[0]["summary_text"])

# 요약에 대한 감성 분류
sentiment = classifier(summary[0]["summary_text"])
print("\n\n--- 요약 대상 감성분석 결과 ---\n")
print("Sentiment:", sentiment)


Device set to use cpu
Device set to use cpu




--- 원문 대상 감성분석 결과 ---

Sentiment: [{'label': 'NEGATIVE', 'score': 0.9925853610038757}]


--- 텍스트 요약 결과 ---

Summary: The package arrived several days later than the estimated date. There was no clear explanation or advance notice regarding the delay. Customer support was unhelpful when contacted.


--- 요약 대상 감성분석 결과 ---

Sentiment: [{'label': 'NEGATIVE', 'score': 0.9994248151779175}]


- 관찰 포인트  
    - 두 모델의 출력 역할이 명확히 분리되는가?  
    - 요약 후 감성분류 결과가 달라질까?  

#### (2) 실습 2. 요약 → 질의응답 흐름 설계 (개념 실습)  

-  요약 결과를 QA 모델의 context로 사용한다고 가정  
- “이 구조가 왜 필요한가?” 토의  